# Queen Editor — Tek foto (Bölüm 4)

Drive'ı bağlar → repoyu klonlar → **ComfyUI'yi kurar** (8 custom node + ~7.5 GiB model) → **Flask**
arayüzü servis eder → **cloudflared** linki basar. Projeye girip prompt yazınca ComfyUI bir foto
üretir, `MyDrive/queenEditor/<proje>/0_a.png` olarak Drive'a düşer ve ekranda görünür.

> **Runtime → Change runtime type → T4 GPU** gerekiyor (SDXL). CPU runtime'da kurulum hücresi durur.

## Kullanım
1. Bu `app.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. **🔑 Secrets** panelinde iki secret olmalı: `GITHUB_TOKEN` (fine-grained, yalnız bu repo,
   `Contents: read`) ve `CIVITAI_COOKIE` (civitai.red → giriş yap → F12 → Application → Cookies →
   `__Secure-civ-token` değeri; ~30 günde bir yenilenir).
3. **Runtime → Run all** → Drive izni ver → ilk kurulum ~10-15 dk → en alttaki linke gir.

In [ ]:
# === CONFIG ===
# The GitHub token comes from Colab's Secrets store (🔑 in the left sidebar), NOT this cell
# -- set once per Google account, never pasted again, never in the notebook source or git.
# Add a secret named GITHUB_TOKEN (fine-grained, this repo, "Contents: read") and grant this
# notebook access. See README for the token setup.
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""   # secret missing or access not granted -> the assert below explains the fix

BRANCH       = "feat/queen-editor-v2"       # dev branch for now; switch to "main" after merge
REPO         = "AltanBaysal/Internal-tools" # <owner>/<repo>
CLONE_DIR    = "/content/Internal-tools"    # clone target on Colab's local disk
APP_PORT     = 8000                         # Flask port (matches backend/config.py)
DRIVE_FOLDER = "queenEditor"                # proje kökü (MyDrive altında) — adı buradan değiştir

# === ComfyUI (kurulum + üretim; backend QE_COMFY_URL ile bu adrese konuşur) ===
COMFY_PORT  = 8188
COMFY_ROOT  = "/content/ComfyUI"
COMFY_LOG   = "/content/comfyui.log"
COMFYUI_URL = f"http://127.0.0.1:{COMFY_PORT}"

# Civitai's gated models need the session cookie. Like GITHUB_TOKEN it comes from Colab Secrets --
# this notebook is committed, so a pasted session JWT would land in git.
try:
    COOKIE_VALUE = userdata.get("CIVITAI_COOKIE")
except Exception:
    COOKIE_VALUE = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu notebook'a erişimi aç (fine-grained, yalnız bu repo, Contents: read)."
)
assert len(COOKIE_VALUE or "") > 200, (
    "❌ CIVITAI_COOKIE yok/çok kısa — Colab 🔑 Secrets'a 'CIVITAI_COOKIE' adıyla ekle: "
    "civitai.red → giriş → F12 → Application → Cookies → __Secure-civ-token değeri (ES256 JWT)"
)

# SDXL needs a GPU; on a CPU runtime the model download would finish and ComfyUI would then fail.
# A CPU runtime has no driver at all, so nvidia-smi is missing rather than failing.
import subprocess as _sp
try:
    _gpu = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    _gpu_name = _gpu.stdout.strip() if _gpu.returncode == 0 else ""
except FileNotFoundError:
    _gpu_name = ""
assert _gpu_name, (
    "❌ GPU yok — Runtime → Change runtime type → T4 GPU seç ve Run all'ı yeniden çalıştır"
)

print(f"✓ GPU: {_gpu_name}")
print("✓ CONFIG hazır (token Colab Secrets'tan okundu)")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")
print(f"✓ Proje kökü: MyDrive/{DRIVE_FOLDER}")

In [ ]:
# === Mount Google Drive ===
# Projects ARE Drive folders, so the mount must succeed before the server starts: writing under
# /content/drive without a mount silently lands on Colab's local disk, and those folders die with
# the runtime. The first run opens a Google permission window -- grant it.
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # first run creates it; later runs reuse it
assert os.path.isdir(DRIVE_ROOT), f"❌ Proje kökü oluşmadı: {DRIVE_ROOT}"
print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")

In [ ]:
# === Clone (delete-and-reclone: the local tree is disposable, always fetch the latest) ===
# subprocess.run with an argument LIST (not shell=True): the token never reaches the shell
# history or a log line. On failure git's stderr is printed RAW, with the token masked.
import os, shutil, subprocess

def _mask(text):
    """Replace the token with <token> so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)              # no pull/merge -- a fresh clone has one behaviour

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"   # never printed (carries the token)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Raw git output, token masked -- never invent a cause (repo comment rule).
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The built frontend ships in the repo (frontend/dist) -- fail loud if it is missing, so a
# forgotten rebuild-and-commit shows up here, not as a blank page.
DIST = os.path.join(CLONE_DIR, "queen-editor", "frontend", "dist", "index.html")
assert os.path.exists(DIST), f"❌ Derlenmiş arayüz yok: {DIST} — frontend'i derleyip commit'le (README)"

# The graph ships with the repo too (our own copy) -- a forgotten commit shows up here.
WORKFLOW = os.path.join(CLONE_DIR, "queen-editor", "workflow_api.json")
assert os.path.exists(WORKFLOW), f"❌ Grafik yok: {WORKFLOW} — workflow_api.json commit'lenmiş mi?"
print("✓ Klon tamam (derlenmiş arayüz + grafik mevcut)")

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
# Used by section 3 (custom nodes), 4 (model download) and 6 (render); defined once (DRY).
import os, json, time, struct, subprocess

# Sections 2 and 3 do not need CONFIG (ComfyUI installs to a hardcoded local path, no Drive), so
# without this gate a failed CONFIG cell stays invisible until section 4 -- after a ~5 min install.
assert "COMFY_ROOT" in globals(), "❌ Önce 1) CONFIG hücresini çalıştır — Drive mount edilmemiş, PROMPT okunmamış"

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for download failures: the downloader exits non-zero on an HTTP error, on a
    transfer that ends before the announced length, and on a full disk.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (HF's Xet CDN answers HEAD with 403
    while serving the GET fine, so a HEAD-based size check reads the error body as the size).

    ok      -> header parses and the file is exactly as long as its header says
    partial -> valid prefix, shorter than expected: safe to resume
    invalid -> empty / error page / longer than expected: garbage, stop
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## ComfyUI + Custom Node'lar (8)

Grafiğin ihtiyacı olan 7 paket + Manager. Liste grafiğin node künyelerinden çıkarıldı; kalan node'lar
comfy-core, kurulum istemez. Biri başarısız olursa hücre `RuntimeError` ile durur (fail-loud).

In [ ]:
%cd /content

# === System deps + ComfyUI ===
!apt-get install -y aria2 > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides to the Basic V37 graph
CUSTOM_NODES = [
    ("ComfyUI-Manager",           "https://github.com/ltdrdata/ComfyUI-Manager.git"),          # detect missing nodes in the UI
    ("rgthree-comfy",             "https://github.com/rgthree/rgthree-comfy.git"),             # Power Lora Loader, Seed, Fast Groups Bypasser, Image Comparer
    ("ComfyUI-Impact-Pack",       "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),      # FaceDetailer, wildcard prompts, SAMLoader, switches
    ("ComfyUI-Impact-Subpack",    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git"),   # UltralyticsDetectorProvider
    ("ComfyUI-Easy-Use",          "https://github.com/yolain/ComfyUI-Easy-Use.git"),           # easy int/float, easy hiresFix
    ("ComfyUI-Custom-Scripts",    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),  # MathExpression
    ("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git"),   # UltimateSDUpscale (tiled Remacri upscale)
    ("ComfyUI-KJNodes",           "https://github.com/kijai/ComfyUI-KJNodes.git"),             # ImageResizeKJv2
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    # --recurse-submodules: UltimateSDUpscale vendors its upstream repo as a git submodule;
    # an empty submodule folder makes the node import fail. Harmless for the others.
    run(["git", "clone", "--depth", "1", "--recurse-submodules", url, name], f"clone {name}", timeout=180)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## Modeller — önce gated probe, sonra indir (~7.5 GiB)

Gated erişim **ağır indirmeden önce** doğrulanır (ilk 1 KB): cookie ölmüşse 6.5 GiB'lık checkpoint'e
başlamadan, Civitai'nin **gerçek yanıtıyla** durur. Bozuk/eksik dosyada hücre durur; bozuk dosya
silinmez, inceleme için diskte kalır. Dosyalar grafiğin beklediği adlarla iner.

> **Model eklemek:** aşağıdaki `CIVITAI_MODELS` (ya da açık indirmeler için `OPEN_MODELS`) listesine
> bir satır ekle, yeter — checkpoint klasörüne inen her `.safetensors` arayüzdeki **Model** listesinde
> kendiliğinden görünür. Uygulama hangi modellerin kurulu olduğunu bilmez, ComfyUI'ye sorar; o yüzden
> burada ikinci bir liste tutmaya gerek yok.

In [ ]:
import os, glob

# === Target folders ===
COMFY = COMFY_ROOT
CKPT = f"{COMFY}/models/checkpoints"
LORA = f"{COMFY}/models/loras"
UPSC = f"{COMFY}/models/upscale_models"
BBOX = f"{COMFY}/models/ultralytics/bbox"   # UltralyticsDetectorProvider lists files as "bbox/<name>"
SAMS = f"{COMFY}/models/sams"
for d in [CKPT, LORA, UPSC, BBOX, SAMS]:
    os.makedirs(d, exist_ok=True)

def check_binary(path, min_bytes):
    """State of a .pt/.pth model -> ("ok" | "partial" | "invalid", msg).

    Torch pickle/zip files carry no self-describing total length (unlike safetensors), so the
    honest cheap checks are: the head is not an HTML/JSON error page, and the size clears a loose
    floor (guards against truncated/error downloads, not an exact size). Below the floor counts
    as "partial" so an interrupted download stays resumable.
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(16)
    if head[:1] in (b"<", b"{"):
        return "invalid", f"error page? ({human(size)})"
    if size < min_bytes:
        return "partial", f"{human(size)} < taban {human(min_bytes)}"
    return "ok", human(size)

# === Single download function — shared flow for HF (aria2c) and Civitai (curl) (DRY) ===
def fetch(url, target_dir, filename, label, *, parallel, headers=None, validate=None):
    """Download + validate a model; anything invalid stops the run (fail-loud, nothing deleted).

    parallel=True -> aria2c (fast for large HF files), False -> curl (Civitai, login cookie).
    validate -> (path) -> (state, msg); default check_safetensors. .pt/.pth files pass a
    check_binary lambda because they have no self-describing length.
    Downloads land in <target>.part and are renamed only once the validator says "ok", so
    ComfyUI never sees a half-written file under the real model name.

    On failure the raw HTTP exchange is printed, not a summary of it: curl runs with
    --fail-with-body (non-zero exit, but the response body is kept instead of discarded) and -D
    (every response header of the redirect chain), so a Civitai 401/403 shows the server's own
    headers and body verbatim.
    """
    validator = validate or check_safetensors
    target = os.path.join(target_dir, filename)
    part = target + ".part"
    hdrs = f"/tmp/{filename}.headers"

    if os.path.exists(target):
        state, msg = validator(target)
        if state == "ok":
            log(f"{label}: zaten var ({msg})")
            return
        raise RuntimeError(f"{label}: {state} — {msg}\n{target}\n--- file head ---\n{head_text(target)}")

    resume = False
    if os.path.exists(part):
        state, msg = validator(part)
        if state == "invalid":
            # Resuming onto garbage would append good bytes to it and hide the problem.
            raise RuntimeError(f"{label}: .part {state} — {msg}\n{part}\n--- file head ---\n{head_text(part)}")
        if state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
            resume = True

    if not os.path.exists(part) or resume:
        log(f"{label}: iniyor")
        if parallel:
            cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--continue=true",
                   "--console-log-level=warn", "--auto-file-renaming=false",
                   "--allow-overwrite=true", "-d", target_dir, "-o", os.path.basename(part)]
            if headers:
                cmd += ["--header", headers]
        else:
            cmd = ["curl", "-L", "-C", "-", "--fail-with-body", "--max-time", "1800",
                   "-D", hdrs, "-o", part]
            if headers:
                cmd += ["-H", headers]
        cmd.append(url)
        try:
            run(cmd, label, timeout=3600)
        except RuntimeError as e:
            raise RuntimeError(
                f"{e}\n{url.split('?')[0]}\n"
                f"--- response headers ---\n{head_text(hdrs)}\n"
                f"--- response body ---\n{head_text(part)}"
            ) from None

    state, msg = validator(part)
    if state != "ok":
        raise RuntimeError(f"{label}: {state} — {msg}\n{part}\n{url.split('?')[0]}\n"
                           f"--- response headers ---\n{head_text(hdrs)}\n"
                           f"--- file head ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

# Civitai auth: session cookie ONLY. A ?token= API key authenticates the request as that key's
# account -> creator-gated assets answer 401.
# Host = civitai.RED: the cookie is same-origin there. Sending it to .com is cross-domain and
# returns the login+turnstile page instead of the file.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

def civitai_probe(version_id, label):
    """Fail-fast: range-download the first 1KB to verify gated access BEFORE the 6.5GiB checkpoint.
    On non-2xx or a login wall, surface Civitai's ACTUAL response body -- no hardcoded guesses.
    """
    out = "/content/_probe.bin"
    code = (run(["curl", "-sL", "--max-time", "60", "-r", "0-1023",
                 "-H", cookie_header(), "-w", "%{http_code}", "-o", out,
                 civitai_url(version_id)], f"probe {label}") or "").strip()[-3:]
    body = b""
    if os.path.exists(out):
        with open(out, "rb") as f:
            body = f.read(512)
        os.remove(out)
    # success = 2xx AND the body is real binary (safetensors), not an HTML/JSON error page
    if code.startswith("2") and not body.startswith(b"<") and not body.startswith(b'{"'):
        log(f"{label}: erişim OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: HTTP {code} — Civitai yanıtı: "
                       f"{body.decode('utf-8', 'replace').strip() or '(boş gövde — binary değil)'}")

# === Civitai gated models (curl + login cookie) ===
CIVITAI_MODELS = [
    # (version_id, target_dir, filename, label)
    (2744564, CKPT, "nova3DCGXL_ilV90.safetensors",               "Nova 3DCG XL IL v9.0"),
    (1552087, LORA, "USNR_STYLE_ILL_V1_lokr3-000024.safetensors", "USNR STYLE ILL v1.0"),
]

# === Open downloads (aria2c, no auth) ===
# The graph's default-ON FaceDetailer branch loads the detector + SAM at startup; the bypassed
# Ultimate SD Upscale branch reads Remacri the moment the user enables it -> all three ship ready.
OPEN_MODELS = [
    # (url, target_dir, filename, label, min_bytes)
    ("https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth",
     UPSC, "4x_foolhardy_Remacri.pth", "Remacri 4x upscaler", 50_000_000),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov9c.pt",
     BBOX, "face_yolov9c.pt", "Yuz dedektoru (yolov9c)", 40_000_000),
    ("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
     SAMS, "sam_vit_b_01ec64.pth", "SAM ViT-B", 300_000_000),
]

# 1) Fail-fast: verify gated access before spending ~20 minutes on downloads
log(f"Gated probe: {len(CIVITAI_MODELS)} asset")
for vid, d, fn, label in CIVITAI_MODELS:
    civitai_probe(vid, label)

# 2) Open models (aria2c)
for url, d, fn, label, floor in OPEN_MODELS:
    fetch(url, d, fn, label, parallel=True, validate=lambda p, m=floor: check_binary(p, m))

# 3) Civitai — parallel=False: aria2c forwards the cookie to the B2 store on redirect and gets 403,
#    curl drops it cross-host and gets through.
for vid, d, fn, label in CIVITAI_MODELS:
    fetch(civitai_url(vid), d, fn, label, parallel=False, headers=cookie_header())

# === Summary (reaching here means everything downloaded + validated) ===
for title, folder, pattern in [("checkpoints", CKPT, "*.safetensors"), ("loras", LORA, "*.safetensors"),
                               ("upscale_models", UPSC, "*.pth"), ("ultralytics/bbox", BBOX, "*.pt"),
                               ("sams", SAMS, "*.pth")]:
    print(f"\n📂 {title}/")
    for f in sorted(glob.glob(f"{folder}/{pattern}")):
        print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## ComfyUI'yi başlat (arka planda)

ComfyUI subprocess olarak kalkar; arayüzün backend'i `QE_COMFY_URL` ile bu adrese konuşur. **90 sn
içinde hazır olmazsa** hücre log'un son 30 satırını basıp durur — sonraki hücre ölü sunucuya
çalışmasın. Tünel yok: ComfyUI'ın kendi arayüzü açılmıyor.

In [ ]:
import subprocess, time, os, urllib.request

# Re-run safety: kill the previous instance before starting a new one
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

# === Start in background (logs to file) ===
# No --enable-manager: nothing opens the UI here, and a missing node already failed loudly during
# install. No tunnel either -- the render cell talks to localhost.
comfy_log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_ROOT, stdout=comfy_log, stderr=subprocess.STDOUT,
)
log(f"ComfyUI başlatıldı (PID {proc.pid}), log: {COMFY_LOG}")

# === Ready? max 90s — otherwise fail-loud with the server's own log ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"{COMFYUI_URL}/system_stats", timeout=2)
        log(f"ComfyUI hazır ({(i + 1) * 2}s)", "OK")
        break
    except Exception:
        pass
else:
    with open(COMFY_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")

In [ ]:
# === Start Flask (background) + cloudflared tunnel ===
# Flask serves the pre-built frontend/dist and /api. It runs as a module from queen-editor/ so
# `backend` resolves as a package. The cell stays OPEN (tail -f): if it ends, Colab calls the
# runtime idle and kills the tunnel. No npm/build here -- the UI ships built (ComfyUI pattern).
import subprocess, time, os, re, urllib.request

APP_DIR = os.path.join(CLONE_DIR, "queen-editor")
FLASK_LOG = "/content/flask.log"

# Re-run safety: kill previous instances before starting new ones
subprocess.run(["pkill", "-f", "backend.main"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(FLASK_LOG, "w")
# The backend reads its Drive root and its ComfyUI address from the environment
# (backend/config.py) -- both are decided in the cells above, not hardcoded in the app.
flask_env = {**os.environ, "QE_DRIVE_ROOT": DRIVE_ROOT, "QE_COMFY_URL": COMFYUI_URL}
subprocess.Popen(["python", "-m", "backend.main"], cwd=APP_DIR, env=flask_env,
                 stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(FLASK_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Flask 90 sn içinde /api/health'e cevap vermedi — yukarıdaki log'a bak")
print(f"✓ Flask ayakta ({(i + 1) * 2}s)")

if not os.path.isfile("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunlog):
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read())
        if m:
            link = m.group(0)
            break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 Queen Editor: {link}\n")
print("⬆️  Linke gir → projeye tıkla → prompt yaz → Üret.\n")
print("📡 Sunucu çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", FLASK_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — Flask hâlâ arka planda (yeni link için tekrar çalıştır).")